In [2]:
# 05_significance_tests.ipynb -- Paired significance and equivalence tests on per-file reconstruction recovery, replacing the overlapping-CI reasoning.
# Uses the exported per-file recovery values (no new LLM calls). Tests whether masking "saturates" (C2 vs C3 vs C4) per adversary.
# Output paths are intentionally not printed (anonymity).

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 120, "font.size": 11,
                     "axes.edgecolor": "0.2", "grid.color": "0.85", "image.cmap": "Greys"})
GRAYS = ["0.10", "0.35", "0.60"]

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA, FIG, TAB = ROOT / "data", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)

samples = pd.read_csv(DATA / "risk_samples.csv")  # columns: tier, model, file_id, recovery
MODELS = ["weak", "mid", "strong"]
PAIRS = [("C2", "C3"), ("C3", "C4"), ("C2", "C4")]
DELTA = 0.05        # pre-registered practical-equivalence margin on mean recovery
B = 10000           # bootstrap resamples
ALPHA = 0.05

def paired(model, ta, tb):
    a = samples[(samples.model == model) & (samples.tier == ta)].sort_values("file_id")["recovery"].to_numpy()
    b = samples[(samples.model == model) & (samples.tier == tb)].sort_values("file_id")["recovery"].to_numpy()
    return a, b

def boot_ci_mean_diff(d):
    idx = rng.integers(0, len(d), size=(B, len(d)))
    means = d[idx].mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

rows = []
for model in MODELS:
    for ta, tb in PAIRS:
        a, b = paired(model, ta, tb)
        d = a - b
        mean_d = float(d.mean())
        lo, hi = boot_ci_mean_diff(d)
        # Wilcoxon signed-rank (two-sided); guard the all-zero-difference case.
        if np.allclose(d, 0):
            w_p = 1.0
        else:
            w_p = float(stats.wilcoxon(a, b, zero_method="wilcox", correction=False,
                                       alternative="two-sided").pvalue)
        # Paired effect size: Cohen's dz.
        dz = mean_d / d.std(ddof=1) if d.std(ddof=1) > 0 else 0.0
        # TOST equivalence against +/- DELTA (two one-sided t-tests on paired diffs).
        n = len(d); se = d.std(ddof=1) / np.sqrt(n) if n > 1 else np.inf
        if se == 0:
            tost_p = 0.0
        else:
            t_low = (mean_d - (-DELTA)) / se
            t_high = (mean_d - DELTA) / se
            p_low = stats.t.sf(t_low, df=n - 1)      # H0: diff <= -delta
            p_high = stats.t.cdf(t_high, df=n - 1)    # H0: diff >= +delta
            tost_p = float(max(p_low, p_high))
        rows.append({"model": model, "pair": f"{ta}-{tb}", "mean_diff": mean_d,
                     "ci_lo": lo, "ci_hi": hi, "wilcoxon_p": w_p, "cohen_dz": dz,
                     "tost_p": tost_p})

res = pd.DataFrame(rows)
# Holm-Bonferroni across the family of Wilcoxon tests.
m = len(res)
order = res["wilcoxon_p"].argsort().to_numpy()
holm = np.empty(m)
prev = 0.0
for rank, i in enumerate(order):
    adj = min(1.0, (m - rank) * res["wilcoxon_p"].iloc[i])
    prev = max(prev, adj)
    holm[i] = prev
res["wilcoxon_p_holm"] = holm
res["different_holm"] = res["wilcoxon_p_holm"] < ALPHA
res["equivalent_tost"] = res["tost_p"] < ALPHA

def verdict(r):
    if r["different_holm"] and not r["equivalent_tost"]:
        return "different"
    if r["equivalent_tost"] and not r["different_holm"]:
        return "equivalent"
    if r["equivalent_tost"] and r["different_holm"]:
        return "different-but-small"
    return "inconclusive"
res["verdict"] = res.apply(verdict, axis=1)
res.to_csv(TAB / "significance_tests.csv", index=False)

# Minimum detectable effect (paired, power 0.8, alpha 0.05, two-sided) at this N, using C2-C3 strong sd.
from math import sqrt
za, zb = stats.norm.ppf(1 - ALPHA / 2), stats.norm.ppf(0.8)
mde_rows = []
for model in MODELS:
    a, b = paired(model, "C2", "C3")
    sd = (a - b).std(ddof=1)
    n = len(a)
    mde = (za + zb) * sd / sqrt(n)
    mde_rows.append({"model": model, "n": n, "sd_paired_diff": sd, "mde_mean_diff": mde})
pd.DataFrame(mde_rows).to_csv(TAB / "power_mde.csv", index=False)

# --- Figure: paired mean differences with 95% bootstrap CIs, equivalence band ---
fig, ax = plt.subplots(figsize=(5.8, 3.6))
pair_labels = ["C2-C3", "C3-C4", "C2-C4"]
x = np.arange(len(pair_labels))
for i, model in enumerate(MODELS):
    sub = res[res.model == model].set_index("pair").loc[pair_labels]
    y = sub["mean_diff"].to_numpy()
    lo = sub["ci_lo"].to_numpy(); hi = sub["ci_hi"].to_numpy()
    yerr = np.vstack([y - lo, hi - y])
    ax.errorbar(x + (i - 1) * 0.12, y, yerr=yerr, fmt=["o", "s", "^"][i], color=GRAYS[i],
                capsize=3, linewidth=0, elinewidth=1.3, label=model)
ax.axhspan(-DELTA, DELTA, color="0.88", alpha=0.7, zorder=0, label=f"equivalence +/-{DELTA}")
ax.axhline(0, color="0.1", linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(pair_labels)
ax.set_xlabel("Tier pair")
ax.set_ylabel("Paired mean difference in recovery")
ax.legend(frameon=False, fontsize=8, ncol=2)
save_fig(fig, "fig_paired_differences")

print("Paired tests (per-file, N=50), delta =", DELTA)
print(res[["model", "pair", "mean_diff", "ci_lo", "ci_hi", "wilcoxon_p_holm", "cohen_dz", "tost_p", "verdict"]].to_string(index=False))
print("\nMinimum detectable effect (power 0.8):")
print(pd.DataFrame(mde_rows).to_string(index=False))


Paired tests (per-file, N=50), delta = 0.05
 model  pair  mean_diff     ci_lo    ci_hi  wilcoxon_p_holm  cohen_dz   tost_p    verdict
  weak C2-C3  -0.001247 -0.020606 0.022991         1.000000 -0.015600 0.000039 equivalent
  weak C3-C4   0.000000  0.000000 0.000000         1.000000  0.000000 0.000000 equivalent
  weak C2-C4  -0.001247 -0.021333 0.023050         1.000000 -0.015600 0.000039 equivalent
   mid C2-C3   0.005017 -0.031519 0.042066         1.000000  0.037422 0.010822 equivalent
   mid C3-C4   0.000000  0.000000 0.000000         1.000000  0.000000 0.000000 equivalent
   mid C2-C4   0.005017 -0.032143 0.040553         1.000000  0.037422 0.010822 equivalent
strong C2-C3   0.065558  0.034199 0.098273         0.002375  0.569785 0.828159  different
strong C3-C4   0.000000  0.000000 0.000000         1.000000  0.000000 0.000000 equivalent
strong C2-C4   0.065558  0.034255 0.097594         0.002375  0.569785 0.828159  different

Minimum detectable effect (power 0.8):
 model  n  sd_pa